# Feature Engineering: Astram Event Data

Creates leakage-safe model-ready features for road closure and duration prediction.

## 1. Imports

Loads lightweight data-processing libraries used for feature preparation.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Load Dataset

Reads the raw Astram event CSV and keeps a copy before transformations.

In [2]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "data").exists()), cwd)
DATA_PATH = PROJECT_ROOT / "data" / "Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

raw_df = pd.read_csv(DATA_PATH, low_memory=False)
df = raw_df.copy()

df.shape

(8173, 46)

## 3. Standardize Column Names

Normalizes column names to make feature creation consistent and less error-prone.

In [3]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

df.columns.tolist()

['id',
 'event_type',
 'latitude',
 'longitude',
 'endlatitude',
 'endlongitude',
 'address',
 'end_address',
 'event_cause',
 'requires_road_closure',
 'start_datetime',
 'end_datetime',
 'status',
 'authenticated',
 'modified_datetime',
 'map_file',
 'direction',
 'description',
 'veh_type',
 'veh_no',
 'corridor',
 'priority',
 'cargo_material',
 'reason_breakdown',
 'age_of_truck',
 'created_date',
 'route_path',
 'client_id',
 'created_by_id',
 'last_modified_by_id',
 'assigned_to_police_id',
 'citizen_accident_id',
 'comment',
 'police_station',
 'meta_data',
 'kgid',
 'resolved_at_address',
 'resolved_at_latitude',
 'resolved_at_longitude',
 'closed_by_id',
 'closed_datetime',
 'resolved_by_id',
 'resolved_datetime',
 'gba_identifier',
 'zone',
 'junction']

## 4. Parse Date-Time Columns

Converts timestamp fields to timezone-aware datetimes and IST versions for time-based features.

In [4]:
datetime_cols = [
    "start_datetime",
    "end_datetime",
    "created_date",
    "modified_datetime",
    "closed_datetime",
    "resolved_datetime",
]

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
        df[f"{col}_ist"] = df[col].dt.tz_convert("Asia/Kolkata")

df[[col for col in ["start_datetime_ist", "end_datetime_ist", "created_date_ist"] if col in df.columns]].head()

,start_datetime_ist,end_datetime_ist,created_date_ist
0,2024-03-07 22:31:48.111000+05:30,NaT,2024-03-07 22:33:51.164032+05:30
1,2024-01-30 09:37:24.173000+05:30,NaT,2024-01-30 09:38:22.954979+05:30
2,2023-11-11 11:48:03.343000+05:30,NaT,2023-11-11 11:50:00.989398+05:30
3,2024-03-07 23:26:55.061000+05:30,NaT,2024-03-07 23:28:56.696892+05:30
4,2024-01-30 10:26:32.348000+05:30,NaT,2024-01-30 10:28:55.937662+05:30


## 5. Clean Basic Categories

Standardizes categorical values so repeated labels are treated consistently.

In [5]:
category_cols = [
    "event_type",
    "event_cause",
    "direction",
    "veh_type",
    "corridor",
    "cargo_material",
    "reason_breakdown",
    "police_station",
    "zone",
    "junction",
]

null_tokens = {"": pd.NA, "nan": pd.NA, "none": pd.NA, "null": pd.NA, "<na>": pd.NA, "na": pd.NA}

for col in category_cols:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
            .replace(null_tokens)
        )

if "event_cause" in df.columns:
    df["event_cause"] = df["event_cause"].replace({
        "fog / low visibility": "fog_low_visibility",
        "traffic congestion": "congestion",
        "vehicle break down": "vehicle_breakdown",
        "vehicle breakdown": "vehicle_breakdown",
    })

if "priority" in df.columns:
    df["priority_clean"] = (
        df["priority"].astype("string").str.strip().str.lower().replace(null_tokens)
    )

if "authenticated" in df.columns:
    df["authenticated_clean"] = (
        df["authenticated"].astype("string").str.strip().str.lower().replace(null_tokens)
    )

df[[col for col in category_cols if col in df.columns]].head()

,event_type,event_cause,direction,veh_type,corridor,cargo_material,reason_breakdown,police_station,zone,junction
0,unplanned,vehicle_breakdown,<NA>,lcv,tumkur road,<NA>,<NA>,peenya,<NA>,<NA>
1,unplanned,vehicle_breakdown,<NA>,heavy_vehicle,orr east 1,<NA>,<NA>,hsr layout,<NA>,<NA>
2,unplanned,others,<NA>,<NA>,non-corridor,<NA>,<NA>,wilson garden,central zone 2,urvashijunction
3,unplanned,tree_fall,<NA>,<NA>,non-corridor,<NA>,<NA>,sadashivanagar,<NA>,<NA>
4,unplanned,vehicle_breakdown,<NA>,private_bus,non-corridor,<NA>,<NA>,wilson garden,<NA>,lalbaghmaingatejunc


## 6. Create Target Columns

Builds the road-closure label and valid duration bands used for supervised modelling.

In [6]:
def parse_bool_target(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("Int64")

    normalized = series.astype("string").str.strip().str.lower()
    parsed = pd.Series(pd.NA, index=series.index, dtype="Int64")
    parsed.loc[normalized.isin({"true", "1", "yes", "y"})] = 1
    parsed.loc[normalized.isin({"false", "0", "no", "n"})] = 0
    return parsed

if "requires_road_closure" in df.columns:
    parsed_closure = parse_bool_target(df["requires_road_closure"])
    df["target_road_closure_missing"] = parsed_closure.isna().astype("int")
    df["target_road_closure"] = parsed_closure.fillna(0).astype("int")

# Outcome timestamps are labels only. They must never be used as forward-looking model features.
duration_source_cols = [col for col in ["end_datetime", "resolved_datetime", "closed_datetime"] if col in df.columns]
duration_frame = pd.DataFrame(index=df.index)

for col in duration_source_cols:
    minutes = (df[col] - df["start_datetime"]).dt.total_seconds() / 60
    valid_minutes = minutes.where(minutes.between(0, 24 * 60))
    feature_name = f"duration_minutes_from_{col.replace('_datetime', '')}"
    df[feature_name] = valid_minutes
    duration_frame[col] = valid_minutes

if not duration_frame.empty:
    df["duration_minutes_raw"] = duration_frame.min(axis=1)
    df["duration_source_count_24h"] = duration_frame.notna().sum(axis=1)
    df["has_duration_label"] = df["duration_minutes_raw"].notna().astype("int")
    df["duration_band"] = pd.cut(
        df["duration_minutes_raw"],
        bins=[0, 60, 240, 24 * 60],
        labels=["short", "medium", "long"],
        include_lowest=True,
    )

df[[col for col in [
    "target_road_closure", "target_road_closure_missing",
    "duration_minutes_raw", "duration_source_count_24h", "duration_band"
] if col in df.columns]].head()

,target_road_closure,target_road_closure_missing,duration_minutes_raw,duration_source_count_24h,duration_band
0,0,0,NaN,0,NaN
1,0,0,10.377589,1,short
2,0,0,NaN,0,NaN
3,1,0,NaN,0,NaN
4,0,0,38.749838,1,short


## 7. Time Features

Extracts calendar, peak-hour, night, cyclic time, and report-lag features.

In [7]:
start_col = "start_datetime_ist"

if start_col in df.columns:
    df["start_hour"] = df[start_col].dt.hour
    df["start_dayofweek"] = df[start_col].dt.dayofweek
    df["start_day_name"] = df[start_col].dt.day_name().str.lower()
    df["start_month"] = df[start_col].dt.strftime("%Y-%m")
    df["start_weekofyear"] = df[start_col].dt.isocalendar().week.astype("Int64")
    df["is_weekend"] = df["start_dayofweek"].isin([5, 6]).astype("int")
    df["is_morning_peak"] = df["start_hour"].between(8, 11).astype("int")
    df["is_evening_peak"] = df["start_hour"].between(17, 20).astype("int")
    df["is_peak_hour"] = ((df["is_morning_peak"] == 1) | (df["is_evening_peak"] == 1)).astype("int")
    df["is_night"] = ((df["start_hour"] >= 22) | (df["start_hour"] <= 5)).astype("int")
    df["hour_sin"] = np.sin(2 * np.pi * df["start_hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["start_hour"] / 24)
    df["day_sin"] = np.sin(2 * np.pi * df["start_dayofweek"] / 7)
    df["day_cos"] = np.cos(2 * np.pi * df["start_dayofweek"] / 7)
    df["peak_period"] = np.select(
        [df["is_morning_peak"] == 1, df["is_evening_peak"] == 1, df["is_night"] == 1],
        ["morning_peak", "evening_peak", "night"],
        default="off_peak",
    )

if {"created_date", "start_datetime"}.issubset(df.columns):
    df["report_lag_minutes"] = (df["created_date"] - df["start_datetime"]).dt.total_seconds() / 60
    df["report_lag_is_negative"] = (df["report_lag_minutes"] < 0).fillna(False).astype("int")
    df["report_lag_minutes_clipped"] = df["report_lag_minutes"].clip(lower=0, upper=24 * 60)
    df["report_lag_hours_clipped"] = df["report_lag_minutes_clipped"] / 60
    df["reporting_delay_minutes"] = df["report_lag_minutes_clipped"]

time_features = [
    "start_hour", "start_dayofweek", "start_day_name", "start_month", "start_weekofyear",
    "is_weekend", "is_peak_hour", "is_night", "peak_period",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "report_lag_minutes", "report_lag_minutes_clipped", "report_lag_hours_clipped",
    "report_lag_is_negative", "reporting_delay_minutes",
]

df[[col for col in time_features if col in df.columns]].head()

,start_hour,start_dayofweek,start_day_name,start_month,start_weekofyear,is_weekend,is_peak_hour,is_night,peak_period,hour_sin,hour_cos,day_sin,day_cos,report_lag_minutes,report_lag_minutes_clipped,report_lag_hours_clipped,report_lag_is_negative,reporting_delay_minutes
0,22.0,3.0,thursday,2024-03,10,0,0,1,night,-0.500000,0.866025,0.433884,-0.900969,2.050884,2.050884,0.034181,0,2.050884
1,9.0,1.0,tuesday,2024-01,5,0,1,0,morning_peak,0.707107,-0.707107,0.781831,0.623490,0.979700,0.979700,0.016328,0,0.979700
2,11.0,5.0,saturday,2023-11,45,1,1,0,morning_peak,0.258819,-0.965926,-0.974928,-0.222521,1.960773,1.960773,0.032680,0,1.960773
3,23.0,3.0,thursday,2024-03,10,0,0,1,night,-0.258819,0.965926,0.433884,-0.900969,2.027265,2.027265,0.033788,0,2.027265
4,10.0,1.0,tuesday,2024-01,5,0,1,0,morning_peak,0.500000,-0.866025,0.781831,0.623490,2.393161,2.393161,0.039886,0,2.393161


## 8. Location Features

Creates coordinate quality, route span, city-distance, and location-grid features.

In [8]:
coordinate_cols = ["latitude", "longitude", "endlatitude", "endlongitude", "resolved_at_latitude", "resolved_at_longitude"]

for col in coordinate_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


def haversine_km(lat1, lon1, lat2, lon2):
    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

if {"latitude", "longitude"}.issubset(df.columns):
    df["valid_start_coordinate"] = (
        df["latitude"].between(12.0, 14.2) & df["longitude"].between(76.5, 78.5)
    ).astype("int")
    df["has_start_location"] = df[["latitude", "longitude"]].notna().all(axis=1).astype("int")
    df["lat_round_3"] = df["latitude"].where(df["valid_start_coordinate"].eq(1)).round(3)
    df["lon_round_3"] = df["longitude"].where(df["valid_start_coordinate"].eq(1)).round(3)
    df["location_grid"] = df["lat_round_3"].astype("string") + "_" + df["lon_round_3"].astype("string")
    df["distance_to_city_center_km"] = haversine_km(df["latitude"], df["longitude"], 12.9716, 77.5946)
    df.loc[df["valid_start_coordinate"].eq(0), "distance_to_city_center_km"] = np.nan

if {"endlatitude", "endlongitude"}.issubset(df.columns):
    df["end_point_missing_or_zero"] = (
        df[["endlatitude", "endlongitude"]].isna().any(axis=1) |
        df[["endlatitude", "endlongitude"]].fillna(0).eq(0).any(axis=1)
    ).astype("int")
    df.loc[df["end_point_missing_or_zero"].eq(1), ["endlatitude", "endlongitude"]] = np.nan
    df["has_end_location"] = df[["endlatitude", "endlongitude"]].notna().all(axis=1).astype("int")

if {"latitude", "longitude", "endlatitude", "endlongitude"}.issubset(df.columns):
    df["route_distance_km"] = haversine_km(df["latitude"], df["longitude"], df["endlatitude"], df["endlongitude"])
    df["has_route_span"] = (df["route_distance_km"].fillna(0) > 0.05).astype("int")

location_features = [
    "latitude", "longitude", "valid_start_coordinate", "has_start_location",
    "lat_round_3", "lon_round_3", "location_grid", "distance_to_city_center_km",
    "end_point_missing_or_zero", "has_end_location", "route_distance_km", "has_route_span",
    "corridor", "zone", "junction", "police_station",
]

df[[col for col in location_features if col in df.columns]].head()

,latitude,longitude,valid_start_coordinate,has_start_location,lat_round_3,lon_round_3,location_grid,distance_to_city_center_km,end_point_missing_or_zero,has_end_location,route_distance_km,has_route_span,corridor,zone,junction,police_station
0,13.040004,77.518099,1,1,13.040,77.518,13.04_77.518,11.249443,1,0,NaN,0,tumkur road,<NA>,<NA>,peenya
1,12.921876,77.645158,1,1,12.922,77.645,12.922_77.645,7.783945,1,0,NaN,0,orr east 1,<NA>,<NA>,hsr layout
2,12.955622,77.585708,1,1,12.956,77.586,12.956_77.586,2.021119,1,0,NaN,0,non-corridor,central zone 2,urvashijunction,wilson garden
3,13.006147,77.579435,1,1,13.006,77.579,13.006_77.579,4.178109,0,1,0.013507,0,non-corridor,<NA>,<NA>,sadashivanagar
4,12.953980,77.585233,1,1,12.954,77.585,12.954_77.585,2.206553,1,0,NaN,0,non-corridor,<NA>,lalbaghmaingatejunc,wilson garden


## 9. Text-Based Features

Captures description length, multilingual text signals, and operational keywords.

In [9]:
text_sources = [col for col in ["description", "address", "end_address"] if col in df.columns]

if text_sources:
    combined_text = df[text_sources].fillna("").astype(str).agg(" ".join, axis=1).str.lower()
    description_text = df.get("description", pd.Series("", index=df.index)).fillna("").astype(str)

    df["text_length"] = combined_text.str.len()
    df["description_char_length"] = description_text.str.len()
    df["description_word_count"] = description_text.str.split().str.len()
    df["has_non_ascii_text"] = description_text.str.contains(r"[^\x00-\x7F]", regex=True).astype("int")
    df["has_kannada_text"] = description_text.str.contains(r"[\u0C80-\u0CFF]", regex=True).astype("int")
    df["has_accident_word"] = combined_text.str.contains("accident|crash|collision", regex=True).astype("int")
    df["has_breakdown_word"] = combined_text.str.contains("breakdown|vehicle breakdown|stalled|puncture", regex=True).astype("int")
    df["has_water_word"] = combined_text.str.contains("water|flood|logging|rain", regex=True).astype("int")
    df["has_construction_word"] = combined_text.str.contains("construction|work|repair|cement", regex=True).astype("int")
    df["has_event_word"] = combined_text.str.contains("event|rally|procession|festival|protest", regex=True).astype("int")
    df["has_blocked_word"] = combined_text.str.contains("block|blocked|closure|closed|obstruction|barricade", regex=True).astype("int")
    df["has_jam_word"] = combined_text.str.contains("jam|congestion|traffic|slow|pile", regex=True).astype("int")
    df["has_vip_word"] = combined_text.str.contains("vip|minister|convoy|movement", regex=True).astype("int")
    df["has_location_hint_word"] = combined_text.str.contains("junction|circle|road|main|cross|signal|bridge|gate|layout", regex=True).astype("int")
    df["has_emergency_word"] = combined_text.str.contains("emergency|ambulance|fire|injury|injured|death", regex=True).astype("int")

text_features = [
    "text_length", "description_char_length", "description_word_count",
    "has_non_ascii_text", "has_kannada_text", "has_accident_word", "has_breakdown_word",
    "has_water_word", "has_construction_word", "has_event_word",
    "has_blocked_word", "has_jam_word", "has_vip_word",
    "has_location_hint_word", "has_emergency_word",
]

df[[col for col in text_features if col in df.columns]].head()

,text_length,description_char_length,description_word_count,has_non_ascii_text,has_kannada_text,has_accident_word,has_breakdown_word,has_water_word,has_construction_word,has_event_word,has_blocked_word,has_jam_word,has_vip_word,has_location_hint_word,has_emergency_word
0,133,31,7,0,0,0,0,0,0,0,0,0,0,1,0
1,109,16,2,0,0,0,0,0,0,0,0,0,0,1,0
2,220,117,15,1,1,0,0,0,0,0,0,0,0,1,0
3,189,9,2,0,0,0,0,0,0,0,0,0,0,1,0
4,152,48,7,1,1,0,0,0,0,0,0,0,0,1,0


## 10. Event and Vehicle Flags

Creates binary event and vehicle indicators linked to congestion severity.

In [10]:
if "event_type" in df.columns:
    df["is_planned_event"] = (df["event_type"] == "planned").astype("int")

if "event_cause" in df.columns:
    planned_causes = ["public_event", "procession", "vip_movement", "protest"]
    df["is_public_or_vip_event"] = df["event_cause"].isin(planned_causes).astype("int")
    df["is_breakdown_event"] = (df["event_cause"] == "vehicle_breakdown").astype("int")
    df["is_accident_event"] = (df["event_cause"] == "accident").astype("int")
    df["is_weather_or_visibility_event"] = df["event_cause"].isin(["water_logging", "fog_low_visibility"]).astype("int")
    df["is_road_condition_event"] = df["event_cause"].isin(["pot_holes", "road_conditions", "debris", "construction"]).astype("int")

if "veh_type" in df.columns:
    veh_text = df["veh_type"].fillna("").astype(str)
    df["has_vehicle_type"] = df["veh_type"].notna().astype("int")
    df["is_truck"] = veh_text.str.contains("truck|lorry|hgv|heavy", regex=True).astype("int")
    df["is_bus"] = veh_text.str.contains("bus", regex=True).astype("int")
    df["is_two_wheeler"] = veh_text.str.contains("bike|two|2|scooter|motor", regex=True).astype("int")
    df["is_heavy_vehicle"] = ((df["is_truck"] == 1) | (df["is_bus"] == 1)).astype("int")

if "cargo_material" in df.columns:
    df["has_cargo_material"] = df["cargo_material"].notna().astype("int")

if "age_of_truck" in df.columns:
    df["age_of_truck"] = pd.to_numeric(df["age_of_truck"], errors="coerce")
    df["truck_age_missing"] = df["age_of_truck"].isna().astype("int")
    df["truck_age_band"] = pd.cut(
        df["age_of_truck"],
        bins=[-1, 5, 10, 20, np.inf],
        labels=["new", "mid", "old", "very_old"],
    ).astype("string")

event_vehicle_features = [
    "is_planned_event", "is_public_or_vip_event", "is_breakdown_event",
    "is_accident_event", "is_weather_or_visibility_event", "is_road_condition_event",
    "has_vehicle_type", "is_truck", "is_bus", "is_two_wheeler", "is_heavy_vehicle",
    "has_cargo_material", "age_of_truck", "truck_age_missing", "truck_age_band",
]

df[[col for col in event_vehicle_features if col in df.columns]].head()

,is_planned_event,is_public_or_vip_event,is_breakdown_event,is_accident_event,is_weather_or_visibility_event,is_road_condition_event,has_vehicle_type,is_truck,is_bus,is_two_wheeler,is_heavy_vehicle,has_cargo_material,age_of_truck,truck_age_missing,truck_age_band
0,0,0,1,0,0,0,1,0,0,0,0,0,NaN,1,<NA>
1,0,0,1,0,0,0,1,1,0,0,1,0,NaN,1,<NA>
2,0,0,0,0,0,0,0,0,0,0,0,0,NaN,1,<NA>
3,0,0,0,0,0,0,0,0,0,0,0,0,NaN,1,<NA>
4,0,0,1,0,0,0,1,0,1,0,1,0,NaN,1,<NA>


## 11. Historical Frequency and Risk Features

Adds past-only counts and closure rates to capture recurring risk without future leakage.

In [11]:
if {"event_cause", "corridor"}.issubset(df.columns):
    df["event_cause_corridor"] = df["event_cause"].fillna("unknown") + "_" + df["corridor"].fillna("unknown")

if "start_datetime" in df.columns:
    df = df.sort_values("start_datetime", na_position="last").reset_index(drop=True)

def add_past_count(data, group_col):
    if group_col in data.columns:
        data[f"past_count_{group_col}"] = data.groupby(group_col, dropna=False).cumcount()

history_group_cols = [
    "event_cause", "corridor", "zone", "junction", "police_station", "location_grid", "event_cause_corridor",
]

for col in history_group_cols:
    add_past_count(df, col)

if "target_road_closure" in df.columns:
    global_prior = df["target_road_closure"].mean()

    def add_past_closure_rate(data, group_col):
        if group_col in data.columns:
            grouped = data.groupby(group_col, dropna=False)["target_road_closure"]
            prior_sum = grouped.cumsum() - data["target_road_closure"]
            prior_count = grouped.cumcount()
            data[f"past_closure_rate_{group_col}"] = (prior_sum / prior_count).replace([np.inf, -np.inf], np.nan)
            data[f"past_closure_rate_{group_col}"] = data[f"past_closure_rate_{group_col}"].fillna(global_prior)

    for col in history_group_cols:
        add_past_closure_rate(df, col)

history_features = [col for col in df.columns if col.startswith("past_count_") or col.startswith("past_closure_rate_")]
df[history_features].head()

,past_count_event_cause,past_count_corridor,past_count_zone,past_count_junction,past_count_police_station,past_count_location_grid,past_count_event_cause_corridor,past_closure_rate_event_cause,past_closure_rate_corridor,past_closure_rate_zone,past_closure_rate_junction,past_closure_rate_police_station,past_closure_rate_location_grid,past_closure_rate_event_cause_corridor
0,0,0,0,0,0,0,0,0.082711,0.082711,0.082711,0.082711,0.082711,0.082711,0.082711
1,0,0,0,0,0,0,0,0.082711,0.082711,0.082711,0.082711,0.082711,0.082711,0.082711
2,0,1,1,0,0,0,0,0.082711,0.000000,0.000000,0.082711,0.082711,0.082711,0.082711
3,0,2,0,0,0,0,0,0.082711,0.000000,0.082711,0.082711,0.082711,0.082711,0.082711
4,0,0,0,0,0,0,0,0.082711,0.082711,0.082711,0.082711,0.082711,0.082711,0.082711


## 12. Interaction Features

Combines cause, location, time, and vehicle signals to model context-specific risk.

In [12]:
if {"event_cause", "peak_period"}.issubset(df.columns):
    df["cause_peak_interaction"] = df["event_cause"].fillna("unknown") + "_" + df["peak_period"].astype("string")

if {"event_cause", "zone"}.issubset(df.columns):
    df["zone_cause_interaction"] = df["zone"].fillna("unknown") + "_" + df["event_cause"].fillna("unknown")

if {"event_cause", "corridor"}.issubset(df.columns):
    df["corridor_cause_interaction"] = df["corridor"].fillna("unknown") + "_" + df["event_cause"].fillna("unknown")

if {"event_cause", "is_heavy_vehicle"}.issubset(df.columns):
    df["cause_heavy_vehicle_interaction"] = df["event_cause"].fillna("unknown") + "_heavy_" + df["is_heavy_vehicle"].astype("string")

if {"corridor", "peak_period"}.issubset(df.columns):
    df["corridor_peak_interaction"] = df["corridor"].fillna("unknown") + "_" + df["peak_period"].astype("string")

interaction_features = [
    "cause_peak_interaction",
    "zone_cause_interaction",
    "corridor_cause_interaction",
    "cause_heavy_vehicle_interaction",
    "corridor_peak_interaction",
]

df[[col for col in interaction_features if col in df.columns]].head()

,cause_peak_interaction,zone_cause_interaction,corridor_cause_interaction,cause_heavy_vehicle_interaction,corridor_peak_interaction
0,others_night,central zone 1_others,non-corridor_others,others_heavy_0,non-corridor_night
1,accident_night,central zone 2_accident,cbd 2_accident,accident_heavy_0,cbd 2_night
2,vehicle_breakdown_night,central zone 2_vehicle_breakdown,non-corridor_vehicle_breakdown,vehicle_breakdown_heavy_1,non-corridor_night
3,water_logging_night,south zone 1_water_logging,non-corridor_water_logging,water_logging_heavy_0,non-corridor_night
4,pot_holes_night,south zone 2_pot_holes,irr(thanisandra road)_pot_holes,pot_holes_heavy_0,irr(thanisandra road)_night


## 13. Rare Category Handling

Groups infrequent categories to reduce noise and unstable one-hot columns.

In [13]:
def group_rare_categories(series, min_count=20):
    counts = series.value_counts(dropna=True)
    common_values = counts[counts >= min_count].index
    return series.where(series.isin(common_values), "other_rare")

rare_group_cols = [
    "event_cause",
    "veh_type",
    "direction",
    "corridor",
    "cargo_material",
    "reason_breakdown",
    "police_station",
    "zone",
    "junction",
    "location_grid",
    "truck_age_band",
    "event_cause_corridor",
    "cause_peak_interaction",
    "zone_cause_interaction",
    "corridor_cause_interaction",
    "cause_heavy_vehicle_interaction",
    "corridor_peak_interaction",
]

for col in rare_group_cols:
    if col in df.columns:
        df[f"{col}_grouped"] = group_rare_categories(df[col])

grouped_cols = [col for col in df.columns if col.endswith("_grouped")]
df[grouped_cols].head()

,event_cause_grouped,veh_type_grouped,direction_grouped,corridor_grouped,cargo_material_grouped,reason_breakdown_grouped,police_station_grouped,zone_grouped,junction_grouped,location_grid_grouped,truck_age_band_grouped,event_cause_corridor_grouped,cause_peak_interaction_grouped,zone_cause_interaction_grouped,corridor_cause_interaction_grouped,cause_heavy_vehicle_interaction_grouped,corridor_peak_interaction_grouped
0,others,other_rare,other_rare,non-corridor,other_rare,other_rare,ashok nagar,central zone 1,other_rare,other_rare,other_rare,others_non-corridor,others_night,central zone 1_others,non-corridor_others,others_heavy_0,non-corridor_night
1,accident,other_rare,other_rare,cbd 2,other_rare,other_rare,cubbon park,central zone 2,other_rare,other_rare,other_rare,other_rare,accident_night,other_rare,other_rare,accident_heavy_0,cbd 2_night
2,vehicle_breakdown,bmtc_bus,other_rare,non-corridor,other_rare,other_rare,halasuru gate,central zone 2,other_rare,other_rare,other_rare,vehicle_breakdown_non-corridor,vehicle_breakdown_night,central zone 2_vehicle_breakdown,non-corridor_vehicle_breakdown,vehicle_breakdown_heavy_1,non-corridor_night
3,water_logging,other_rare,other_rare,non-corridor,other_rare,other_rare,jayanagara,south zone 1,other_rare,other_rare,other_rare,water_logging_non-corridor,water_logging_night,other_rare,non-corridor_water_logging,water_logging_heavy_0,non-corridor_night
4,pot_holes,other_rare,other_rare,irr(thanisandra road),other_rare,other_rare,adugodi,south zone 2,koramangalawatertankjunc,other_rare,other_rare,pot_holes_irr(thanisandra road),pot_holes_night,south zone 2_pot_holes,irr(thanisandra road)_pot_holes,pot_holes_heavy_0,irr(thanisandra road)_night


## 14. Select Model Features

Keeps event-time features and excludes post-event or leakage-prone columns.

In [14]:
numeric_feature_cols = [
    "latitude", "longitude", "valid_start_coordinate", "has_start_location", "has_end_location",
    "end_point_missing_or_zero", "route_distance_km", "has_route_span", "distance_to_city_center_km",
    "start_hour", "start_dayofweek", "start_weekofyear", "is_weekend", "is_peak_hour", "is_night",
    "hour_sin", "hour_cos", "day_sin", "day_cos",
    "report_lag_minutes_clipped", "report_lag_hours_clipped", "report_lag_is_negative", "reporting_delay_minutes",
    "text_length", "description_char_length", "description_word_count", "has_non_ascii_text", "has_kannada_text",
    "has_accident_word", "has_breakdown_word", "has_water_word", "has_construction_word",
    "has_event_word", "has_blocked_word", "has_jam_word", "has_vip_word",
    "has_location_hint_word", "has_emergency_word",
    "is_planned_event", "is_public_or_vip_event", "is_breakdown_event", "is_accident_event",
    "is_weather_or_visibility_event", "is_road_condition_event",
    "has_vehicle_type", "is_truck", "is_bus", "is_two_wheeler", "is_heavy_vehicle",
    "has_cargo_material", "age_of_truck", "truck_age_missing",
    "past_count_event_cause", "past_count_corridor", "past_count_zone", "past_count_junction",
    "past_count_police_station", "past_count_location_grid", "past_count_event_cause_corridor",
    "past_closure_rate_event_cause", "past_closure_rate_corridor", "past_closure_rate_zone",
    "past_closure_rate_junction", "past_closure_rate_police_station", "past_closure_rate_location_grid",
    "past_closure_rate_event_cause_corridor",
]

categorical_feature_cols = [
    "event_type",
    "event_cause_grouped",
    "direction_grouped",
    "veh_type_grouped",
    "corridor_grouped",
    "cargo_material_grouped",
    "reason_breakdown_grouped",
    "police_station_grouped",
    "zone_grouped",
    "junction_grouped",
    "location_grid_grouped",
    "truck_age_band_grouped",
    "event_cause_corridor_grouped",
    "peak_period",
    "cause_peak_interaction_grouped",
    "zone_cause_interaction_grouped",
    "corridor_cause_interaction_grouped",
    "cause_heavy_vehicle_interaction_grouped",
    "corridor_peak_interaction_grouped",
    "start_day_name",
    "start_month",
]

leakage_excluded_columns = [
    "status", "modified_datetime", "modified_datetime_ist", "closed_datetime", "closed_datetime_ist",
    "resolved_datetime", "resolved_datetime_ist", "closed_by_id", "resolved_by_id",
    "resolved_at_address", "resolved_at_latitude", "resolved_at_longitude", "last_modified_by_id",
    "duration_minutes_from_end", "duration_minutes_from_resolved", "duration_minutes_from_closed",
    "duration_source_count_24h", "has_duration_label",
]

feature_cols = [col for col in numeric_feature_cols + categorical_feature_cols if col in df.columns]
target_cols = [col for col in ["target_road_closure", "duration_band", "duration_minutes_raw"] if col in df.columns]

feature_table = df[feature_cols + target_cols].copy()
feature_manifest = pd.DataFrame({
    "feature": feature_cols + target_cols + [col for col in leakage_excluded_columns if col in df.columns],
    "feature_type": (
        ["numeric"] * len([col for col in numeric_feature_cols if col in df.columns]) +
        ["categorical"] * len([col for col in categorical_feature_cols if col in df.columns]) +
        ["target"] * len(target_cols) +
        ["excluded_leakage_or_post_event"] * len([col for col in leakage_excluded_columns if col in df.columns])
    ),
})

feature_table.shape, feature_table.columns.tolist()

((8173, 90),
 ['latitude',
  'longitude',
  'valid_start_coordinate',
  'has_start_location',
  'has_end_location',
  'end_point_missing_or_zero',
  'route_distance_km',
  'has_route_span',
  'distance_to_city_center_km',
  'start_hour',
  'start_dayofweek',
  'start_weekofyear',
  'is_weekend',
  'is_peak_hour',
  'is_night',
  'hour_sin',
  'hour_cos',
  'day_sin',
  'day_cos',
  'report_lag_minutes_clipped',
  'report_lag_hours_clipped',
  'report_lag_is_negative',
  'reporting_delay_minutes',
  'text_length',
  'description_char_length',
  'description_word_count',
  'has_non_ascii_text',
  'has_kannada_text',
  'has_accident_word',
  'has_breakdown_word',
  'has_water_word',
  'has_construction_word',
  'has_event_word',
  'has_blocked_word',
  'has_jam_word',
  'has_vip_word',
  'has_location_hint_word',
  'has_emergency_word',
  'is_planned_event',
  'is_public_or_vip_event',
  'is_breakdown_event',
  'is_accident_event',
  'is_weather_or_visibility_event',
  'is_road_condition_

## 15. Handle Missing Values

Fills numeric and categorical gaps so baseline models can train reliably.

In [15]:
model_table = feature_table.copy()

numeric_cols = [col for col in numeric_feature_cols if col in model_table.columns]
categorical_cols = [col for col in categorical_feature_cols if col in model_table.columns]

for col in numeric_cols:
    median_value = model_table[col].median()
    fill_value = 0 if pd.isna(median_value) else median_value
    model_table[col] = model_table[col].fillna(fill_value)

for col in categorical_cols:
    model_table[col] = model_table[col].astype("string").fillna("unknown")

model_table.isna().sum().sort_values(ascending=False).head(10)

duration_band                5409
duration_minutes_raw         5409
valid_start_coordinate          0
has_start_location              0
has_end_location                0
end_point_missing_or_zero       0
route_distance_km               0
has_route_span                  0
latitude                        0
longitude                       0
dtype: int64

## 16. Encode Categorical Features

One-hot encodes categorical features for model-ready numeric training data.

In [16]:
encoded_features = pd.get_dummies(
    model_table[feature_cols],
    columns=[col for col in categorical_cols if col in model_table.columns],
    dummy_na=False,
    dtype=int,
)

road_closure_model_data = None
if "target_road_closure" in model_table.columns:
    road_closure_model_data = encoded_features.copy()
    road_closure_model_data["target_road_closure"] = model_table["target_road_closure"]

duration_model_data = None
if "duration_band" in model_table.columns:
    duration_mask = model_table["duration_band"].notna()
    duration_model_data = encoded_features.loc[duration_mask].copy()
    duration_model_data["duration_band"] = model_table.loc[duration_mask, "duration_band"].astype(str)

encoded_features.shape

(8173, 515)

## 17. Save Feature Outputs

Saves only the two model-ready CSVs needed by the modelling notebooks.

In [17]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

saved_files = []

if road_closure_model_data is not None:
    road_path = OUTPUT_DIR / "model_ready_road_closure.csv"
    road_closure_model_data.to_csv(road_path, index=False)
    saved_files.append(road_path)

if duration_model_data is not None:
    duration_path = OUTPUT_DIR / "model_ready_duration_band.csv"
    duration_model_data.to_csv(duration_path, index=False)
    saved_files.append(duration_path)

saved_files

[WindowsPath('D:/Python/Gridlock/Phase 2/theme 2/outputs/model_ready_road_closure.csv'),
 WindowsPath('D:/Python/Gridlock/Phase 2/theme 2/outputs/model_ready_duration_band.csv')]

## 18. Quick Feature Summary

Checks feature-table shapes and target distributions before modelling.

In [18]:
print("Feature table:", feature_table.shape)
print("Numeric features:", len([col for col in numeric_feature_cols if col in feature_table.columns]))
print("Categorical features:", len([col for col in categorical_feature_cols if col in feature_table.columns]))
if road_closure_model_data is not None:
    print("Road closure model table:", road_closure_model_data.shape)
    print(model_table["target_road_closure"].value_counts(dropna=False))

if duration_model_data is not None:
    print("Duration model table:", duration_model_data.shape)
    print(model_table["duration_band"].value_counts(dropna=False))

Feature table: (8173, 90)
Numeric features: 66
Categorical features: 21
Road closure model table: (8173, 516)
target_road_closure
0    7497
1     676
Name: count, dtype: int64
Duration model table: (2764, 516)
duration_band
NaN       5409
short     1581
medium     903
long       280
Name: count, dtype: int64


## 19. Feature Engineering Summary

The notebook creates leakage-safe targets, time, spatial, text, event, vehicle, historical-risk, and interaction features, then saves only the road-closure and duration model-ready CSVs.